# Building an Eval for an AI Shopping Assistant

In this build-along, you'll learn how to **build an eval** for an AI agent and use it to systematically find and fix problems.

`boutique` is a simple shopping assistant that looks up product prices and does math. It works... mostly. Your job is to figure out exactly where it breaks, why, and how to fix it.

---

**Prepared for Partner Basecamp participants.** Not for reproduction or redistribution as training material — you're free to apply these patterns in your own client work.

### Setup — connect to Claude

Run the next cell. It reads your key from the `.env` file at the repo root and checks it with a real call — you're ready when you see the green **✓ verified** banner. Edited `.env`? Just run the cell again. Red or yellow banner? Do what it says and run the cell again.

**First time on this machine?** From the repo root (the folder that contains `day1` and `day2`), run `python3 basecamp_setup.py` once (Windows: `py basecamp_setup.py`; see `SETUP.md`), then pick the **Anthropic Basecamp (.venv)** kernel (top right).

In [1]:
# ── Setup — connect to Claude ──
# One-time machine setup is basecamp_setup.py at the repo root: run it once, then pick the
# "Anthropic Basecamp (.venv)" kernel. This cell reads your key from .env and checks it with a real
# call. Edited .env? Just run this cell again.
import os
import pathlib
import sys

_root = next((d for d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (d / "basecamp_setup.py").is_file()), None)
if _root is None:
    raise SystemExit("Can't find basecamp_setup.py — open this notebook from inside the "
                     "Basecamp-Exercises folder (see SETUP.md).")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from basecamp_setup import connect

HAIKU = "claude-haiku-5-5"  # this notebook runs Haiku 5.5 only: agent, judge, and connection check
claude = connect(model=HAIKU, fast_model=HAIKU, big_model=HAIKU)
client = claude.client
MODEL, FAST_MODEL, BIG_MODEL = claude.model, claude.fast_model, claude.big_model

import anthropic


## Why Evals?

When you're building with Claude, "try it a few times and see if it works" isn't the best strategy. Vibes are definitely important, but Evals give you:

- **A baseline.** How good is the agent right now? Across what types of queries?
- **A feedback loop.** Change something (prompt, tool design, model), re-run, see if it actually helped and/or if there are any regressions.
- **Confidence.** Before shipping to a customer, you can point to numbers, not just vibes.

Today's build along:

1. **Meet the agent** and begin to notice what works and what doesn't
2. **Define eval tasks** that cover the agent's capabilities and edge cases
3. **Run the eval** and inspect results systematically
4. **Improve the agent** using eval results, then re-run to verify
5. **Add an LLM-as-judge grader** for queries that can't be checked with simple string matching

---

## Part 1: Meet the Agent

`boutique` is a simple single-turn agent. It has two tools:

| Tool | What it does |
|------|-------------|
| `get_product(product)` | Returns the price of an item from the catalog |
| `calculate(op, input1, input2)` | Basic math operations |

The agentic loop is simple:

```
User query → Claude decides what to do → Tool call (if needed) → Tool result → ... → Final answer
```

Skim through the code below. You might need to come back later to inspect:
- The **tool specs** (what Claude sees/knows about each tool)
- The **system prompt**
- The **tool implementations** (what actually happens when a tool is called)

In [2]:
import math
import re
from anthropic import Anthropic
from anthropic.types import ToolUseBlock, TextBlock

# ── Config ────────────────────────────────────────────────────────────────────
#
# This cell holds the IMPROVED agent: variant `v_all_catprompt` from the A/B suite in ab/.
# Measured on Haiku 5.5: 100% on the 11 dev tasks (5 trials) and 100% on the 11 blind holdout
# tasks (3 trials), at ~$0.0004 per query. (In ab/, Sonnet 5.5 scored the same at ~20x the cost,
# so this notebook uses Haiku 5.5 only.)
# The agent exactly as shipped is frozen in ab/shipped/Building_an_Eval.shipped.ipynb, and the
# evidence for every change is in ab/reflection.md.

MODEL = FAST_MODEL  # Haiku 5.5, from the setup cell (alias resolved for your provider)

CATALOG = {
    "jeans": 49.99, "shirt": 29.99, "dress": 59.99, "jacket": 89.99, "sneakers": 74.99,
    "hat": 19.99, "socks": 9.99, "hoodie": 44.99, "shorts": 34.99, "t-shirt": 24.99,
    "sweater": 54.99, "belt": 24.99,
}
CATALOG_NAMES = ", ".join(CATALOG)

# The biggest single lever (ablation: removing it costs the most). Product NAMES only, no prices,
# so get_product stays the one source of truth for prices.
SYSTEM_PROMPT = f"""You are the shopping assistant for Boutique, a small clothing store. Prices are in US dollars.

Boutique sells a fixed catalog of clothing items. It has no brands, sizes, colors or styles to choose between: each item has one price.

- Look up every price with get_product; never quote a price from memory.
- Do arithmetic with calculate, one operation per call; never do math in your head.
- Each message is a complete request. Answer it directly with the information you have, instead of asking follow-up questions about brand, style, currency or budget.
- If a customer asks for something the store doesn't carry, say so plainly. If a catalog item is a close match (for example, a similar kind of clothing), suggest it with its price. Never invent products or prices.

Items Boutique sells: {CATALOG_NAMES}."""

# The harness client: long timeout, standard retries. The verification client above
# uses a short 30s timeout / 1 retry — right for a 1-token ping, too twitchy for a full
# eval run. Re-create it for your provider (Anthropic API, Bedrock, or gateway).
client = claude.make_client(timeout=600.0)

# ── Tool implementations ─────────────────────────────────────────────────────

def _key(s):
    return re.sub(r"[^a-z0-9]", "", s.lower())

_KEYED = {_key(k): k for k in CATALOG}


def get_product(product: str):
    if product in CATALOG:
        return CATALOG[product]
    # Forgiving match: case, punctuation, singular/plural ("TSHIRT", "jean", "hoodies").
    k = _key(product)
    for cand in (k, k + "s", k[:-1] if k.endswith("s") else None, k[:-2] if k.endswith("es") else None):
        if cand and cand in _KEYED:
            return CATALOG[_KEYED[cand]]
    # A miss says what IS available, so the agent can self-correct (shoes -> sneakers).
    return f"Error: '{product}' is not in the catalog. Available items: {CATALOG_NAMES}."


def calculate(op: str, input1: float, input2: float):
    if op == "+": return input1 + input2
    if op == "-": return input1 - input2
    if op == "*": return input1 * input2
    if op == "/":
        if input2 == 0:
            return "Error: division by zero."
        return input1 / input2
    if op == "**": return input1 ** input2
    # Shipped version fell through and returned None, so the agent silently did mental math.
    return f"Error: unsupported op '{op}'. Use one of '+', '-', '*', '/', '**'."

TOOL_REGISTRY = {
    "get_product": get_product,
    "calculate": calculate,
}

# ── Tool specs (sent to Claude) ──────────────────────────────────────────────

GET_PRODUCT_SPEC = {
    "name": "get_product",
    "description": ("Look up the price, in USD, of one catalog item. Pass the catalog item name in "
                    "lowercase singular form as it appears in the catalog (e.g. 'jeans', 't-shirt', "
                    "'hoodie'). Returns the price as a number, or an error if the item isn't sold."),
    "input_schema": {
        "type": "object",
        "properties": {
            "product": {
                "type": "string",
                "description": "Catalog item name, lowercase, e.g. 'jeans' or 't-shirt'.",
            },
        },
        "required": ["product"],
    },
}

CALCULATE_SPEC = {
    "name": "calculate",
    "description": ("Do one arithmetic operation on two numbers and return the result: input1 <op> input2. "
                    "For a percentage discount, multiply: 20% off 89.99 is 89.99 * 0.8."),
    "input_schema": {
        "type": "object",
        "properties": {
            "op": {
                "type": "string",
                "enum": ["+", "-", "*", "/", "**"],
                "description": "One of '+', '-', '*', '/', '**'.",
            },
            "input1": {
                "type": "number",
                "description": "Left operand.",
            },
            "input2": {
                "type": "number",
                "description": "Right operand.",
            },
        },
        "required": ["op", "input1", "input2"],
    },
}

ALL_TOOL_SPECS = [GET_PRODUCT_SPEC, CALCULATE_SPEC]

# ── Agent ─────────────────────────────────────────────────────────────────────

def call_claude(messages, tools, model=None):
    return client.messages.create(
        model=model or MODEL,
        system=SYSTEM_PROMPT,
        max_tokens = 1024,
        tools=tools,
        messages=messages,
    )


def execute_tool(name, inputs):
    try:
        return str(TOOL_REGISTRY[name](**inputs))
    except Exception as e:
        return f"Error: {e}"


def run_agent(prompt, eval_mode=False, model=None):
    messages = [{"role": "user", "content": prompt}]
    total_input_tokens = 0
    total_output_tokens = 0

    while True:
        response = call_claude(messages, tools=ALL_TOOL_SPECS, model=model)
        total_input_tokens += response.usage.input_tokens
        total_output_tokens += response.usage.output_tokens
        messages.append({"role": "assistant", "content": response.content})

        # Break unless Claude asked for a tool. Guarding on "tool_use" (rather than
        # "end_turn") prevents a looping 400 if the model stops for another reason
        # (e.g. max_tokens) — we'd otherwise send back an empty tool_results message.
        # (With server-side tools, also handle stop_reason == "pause_turn".)
        if response.stop_reason != "tool_use":
            break

        tool_calls = [block for block in response.content if isinstance(block, ToolUseBlock)]

        tool_results = []
        for tool_call in tool_calls:
            result = execute_tool(tool_call.name, tool_call.input)
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tool_call.id,
                "content": result,
            })

        messages.append({"role": "user", "content": tool_results})

    if eval_mode:
        return {
            "messages": messages,
            "usage": {"input_tokens": total_input_tokens, "output_tokens": total_output_tokens},
        }

    return "\n".join(block.text for block in response.content if isinstance(block, TextBlock))


print("boutique agent ready.")

boutique agent ready.


### Try It Out

The cell below starts an interactive session with the agent automatically. Type queries and see how it responds. Type `quit` to stop. (Running the whole notebook with Run All? Set `INTERACTIVE_CHAT = False` in that cell first, or just type `quit` when it pauses here.)

Some queries to try:
- `How much do jeans cost?` (simple, should work)
- `Price of a t-shirt?` (will the hyphen trip it up?)
- `How much for shoes?` ("shoes" isn't in the catalog, but "sneakers" is)
- `3 shirts and 2 belts, what's my total?` (multi-tool: lookups + math)
- `What's 20% off a jacket?` (requires percentage math)
- `What do you sell?` (can it describe its own capabilities?)

In [3]:
ANTHROPIC_ORANGE = "#E07A5F"  # brand accent — keeps the boutique agent from blending into the notebook's black-on-white output

def _display_chat_hint():
    """Tells you how to turn on the interactive chat: Anthropic-orange in a notebook, plain
    text when run as a script."""
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or shell.__class__.__name__ != "ZMQInteractiveShell":
            raise RuntimeError("not in a notebook kernel - use the plain-text banner")
        from IPython.display import display, HTML
        display(HTML(
            f'<div style="background:{ANTHROPIC_ORANGE};color:#fff;padding:10px 16px;'
            f'border-radius:8px;font-family:sans-serif;font-size:14px;">'
            f'🛍️ <b>The Boutique Agent</b> is ready — set '
            f'<code style="background:rgba(255,255,255,.3);padding:1px 5px;border-radius:4px;">INTERACTIVE_CHAT = True</code> '
            f'above and run this cell again to chat.</div>'
        ))
    except Exception:
        print("The Boutique Agent is ready — set INTERACTIVE_CHAT = True above and run this cell again to chat.")


INTERACTIVE_CHAT = False  # completed notebook: off so Run All goes straight to the evals — set True to chat with the agent

if not INTERACTIVE_CHAT:
    _display_chat_hint()
else:
    print("Boutique Agent Response Results")

# The input() prompt below is VS Code's own editor UI (like the command palette) — its
# colors always follow the user's VS Code theme and can't be styled from here. Since text
# is the only lever we have, the prompt itself carries the branding instead.
while INTERACTIVE_CHAT:
    query = input("\n🛍️  The Boutique Shopping agent is running, ask your question and press ENTER, press ESC to stop the agent.")
    if not query.strip() or query.strip().lower() in ("quit", "exit", "q"):
        print("Session ended.")
        break
    print(f"\nBoutique: {run_agent(query)}")

---

## Part 2: The Eval Framework

An eval has three main components:

```
Tasks ──> Runner ──> Graders ──> Results
```

- **Tasks** define *what* to test: scenario, expected behavior, and how to grade it
- **Runner** orchestrates execution: sends scenarios' queries to the agent, collects transcripts, applies graders
- **Graders** check *whether* the agent did the right thing and return a score + reason

The runner and graders are defined in the next two cells. Run them both. You can quickly read through the code to understand how they work, but you won't need to modify them (yet).

### Available graders

| Grader | What it checks | Check format |
|--------|---------------|---------------|
| `response_contains` | Final text contains a string (case-insensitive) | `"49.99"` or `"jeans"` |
| `response_numeric` | Final text contains a number within tolerance | `{"value": 49.99, "tolerance": 0.05}` |
| `tool_use` | Agent called a specific tool with specific args | `{"tool_name": "get_product", "arguments": {"product": "jeans"}}` |

Each grader returns a binary score (0 = fail, 1 = pass) and a reason explaining why. A task passes only if **all checks from all graders** pass.

In [4]:
# ── Graders (just run this cell) ──────────────────────────────────────────────

import re

def grade_response_contains(result, check, context=None):
    text = result["final_text"].lower()
    target = check.lower()
    if target in text:
        return {"score": 1.0, "reason": f"Found '{check}' in response"}
    return {"score": 0.0, "reason": f"'{check}' not found in response: {result['final_text'][:200]}"}


def grade_response_numeric(result, check, context=None):
    if isinstance(check, (int, float)):
        value, tolerance = float(check), 0.01
    else:
        value = float(check["value"])
        tolerance = float(check.get("tolerance", 0.01))

    numbers = re.findall(r"-?[\d,]+\.?\d*", result["final_text"])
    for num_str in numbers:
        try:
            num = float(num_str.replace(",", ""))
            if abs(num - value) <= tolerance:
                return {"score": 1.0, "reason": f"Found {num} (expected {value} +/- {tolerance})"}
        except ValueError:
            continue
    return {"score": 0.0, "reason": f"Expected {value} (+/- {tolerance}), found: {numbers[:10]}"}


def grade_tool_use(result, check, context=None):
    tool_name = check["tool_name"]
    expected_args = check.get("arguments", None)

    for call in result["tool_calls"]:
        if call["name"] != tool_name:
            continue
        if expected_args is None:
            return {"score": 1.0, "reason": f"Tool '{tool_name}' was called"}

        # Partial match: only check specified keys
        actual_args = call.get("arguments", {})
        match = all(
            (isinstance(v, str) and isinstance(actual_args.get(k), str) and v.lower() == actual_args[k].lower())
            or actual_args.get(k) == v
            for k, v in expected_args.items()
        )
        if match:
            return {"score": 1.0, "reason": f"Tool '{tool_name}' called with matching args: {expected_args}"}

    actual = [{"name": c["name"], "args": c.get("arguments", {})} for c in result["tool_calls"]]
    if expected_args:
        return {"score": 0.0, "reason": f"'{tool_name}' not called with {expected_args}. Actual: {actual}"}
    return {"score": 0.0, "reason": f"'{tool_name}' never called. Actual: {[c['name'] for c in result['tool_calls']]}"}


GRADER_REGISTRY = {
    "response_contains": grade_response_contains,
    "response_numeric": grade_response_numeric,
    "tool_use": grade_tool_use,
}

print(f"Graders loaded: {list(GRADER_REGISTRY.keys())}")

Graders loaded: ['response_contains', 'response_numeric', 'tool_use']


In [5]:
# ── Eval Runner (just run this cell) ──────────────────────────────────────────

import json, os, time, traceback
from concurrent.futures import ThreadPoolExecutor, as_completed


def parse_transcript(messages):
    """Extract final_text and tool_calls from raw agent transcript."""
    final_text, tool_calls = "", []
    for msg in messages:
        if msg["role"] != "assistant":
            continue
        texts = [block.text for block in msg["content"] if isinstance(block, TextBlock)]
        for block in msg["content"]:
            if isinstance(block, ToolUseBlock):
                tool_calls.append({"name": block.name, "arguments": block.input, "id": block.id})
        # Fix (found in ab/): the shipped loop kept only the LAST text block, so part of an
        # answer could vanish before grading. Grade what run_agent shows the customer: every
        # text block of the final assistant message.
        final_text = "\n".join(texts)
    # Match tool results back to calls
    for msg in messages:
        if msg["role"] != "user" or not isinstance(msg["content"], list):
            continue
        for item in msg["content"]:
            if isinstance(item, dict) and item.get("type") == "tool_result":
                for call in tool_calls:
                    if call["id"] == item["tool_use_id"]:
                        call["result"] = item.get("content", "")
                        break
    return {"final_text": final_text, "tool_calls": tool_calls, "messages": messages}


def run_single_task(agent_fn, task, model=None):
    """Run one task, apply graders, return result with grades + metrics."""
    start = time.time()
    try:
        raw = agent_fn(task["query"], eval_mode=True, model=model)
    except Exception:
        return {
            "task_id": task["id"], "task_description": task.get("description", ""),
            "query": task["query"], "category": task.get("category", ""),
            "error": traceback.format_exc(), "passed": False, "grades": [],
            "metrics": {"time": time.time() - start},
        }

    elapsed = time.time() - start
    result = parse_transcript(raw["messages"])
    usage = raw.get("usage", {})
    turns = sum(1 for m in raw["messages"] if m["role"] == "assistant")
    metrics = {
        "time": round(elapsed, 3), "tool_calls": len(result["tool_calls"]),
        "turns": turns, "input_tokens": usage.get("input_tokens", 0),
        "output_tokens": usage.get("output_tokens", 0),
    }

    grades = []
    context = {"query": task["query"], "task_id": task["id"], "model": model}
    for grader in task.get("graders", []):
        grader_fn = GRADER_REGISTRY.get(grader["type"])
        if grader_fn is None:
            grades.append({"type": grader["type"], "check": None, "score": 0.0, "reason": f"Unknown grader: {grader['type']}"})
            continue
        for check in grader.get("checks", []):
            # A grader that raises (or returns something malformed) fails this one
            # task only — without the guard, the exception would re-raise at
            # f.result() in run_eval and abort the entire run.
            try:
                grade = grader_fn(result, check, context)
                grades.append({"type": grader["type"], "check": check, "score": grade["score"], "reason": grade["reason"]})
            except Exception as exc:
                grades.append({"type": grader["type"], "check": check, "score": 0.0,
                               "reason": f"grader error: {type(exc).__name__}: {exc}"})

    passed = all(g["score"] == 1.0 for g in grades) if grades else False

    return {
        "task_id": task["id"], "task_description": task.get("description", ""),
        "query": task["query"], "category": task.get("category", ""),
        "passed": passed, "grades": grades, "metrics": metrics,
        "final_text": result["final_text"],
        "transcript": [
            block.model_dump() if hasattr(block, "model_dump") else block
            for msg in raw["messages"]
            for block in (msg["content"] if isinstance(msg["content"], list) else [msg["content"]])
        ],
    }


def run_eval(agent_fn, tasks, model=None, num_runs=1, max_workers=5):
    """Run the full eval suite. Returns structured results."""
    all_runs = []
    for _ in range(num_runs):
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(run_single_task, agent_fn, t, model): t for t in tasks}
            run_results = []
            for f in as_completed(futures):
                r = f.result()
                run_results.append(r)
                mark = "PASS" if r["passed"] else ("ERROR" if r.get("error") else "FAIL")
                print(f"  [{len(run_results)}/{len(tasks)}] {r['task_id']}: {mark}", flush=True)
        task_order = {t["id"]: i for i, t in enumerate(tasks)}
        run_results.sort(key=lambda r: task_order.get(r["task_id"], 999))
        all_runs.append(run_results)
    return {"runs": all_runs, "config": {"model": model, "num_runs": num_runs, "num_tasks": len(tasks)}}


def save_results(results, directory="eval_results"):
    """Save eval results to a JSON file."""
    os.makedirs(directory, exist_ok=True)
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    model_name = results["config"].get("model") or "default"
    model_short = model_name.split("-")[1] if "-" in str(model_name) else model_name
    filename = f"{directory}/eval_{model_short}_{timestamp}.json"
    with open(filename, "w") as f:
        json.dump(results, f, indent=2, default=str)
    print(f"Results saved to {filename}")
    return filename


def print_summary(results):
    """Print formatted eval results."""
    config = results["config"]
    print(f"{'=' * 60}")
    print(f"EVAL RESULTS: {config['num_tasks']} tasks, {config['num_runs']} run(s)")
    if config.get("model"): print(f"Model: {config['model']}")
    print(f"{'=' * 60}\n")

    for run_idx, run in enumerate(results["runs"]):
        if config["num_runs"] > 1: print(f"--- Run {run_idx + 1} ---")
        passed = sum(1 for r in run if r["passed"])
        total = len(run)
        print(f"Overall: {passed}/{total} passed ({passed/total*100:.0f}%)\n")

        # Per-category breakdown
        categories = {}
        for r in run:
            cat = r.get("category", "uncategorized")
            categories.setdefault(cat, {"passed": 0, "total": 0})
            categories[cat]["total"] += 1
            if r["passed"]: categories[cat]["passed"] += 1
        if len(categories) > 1:
            print("By category:")
            for cat, c in sorted(categories.items()):
                print(f"  {cat}: {c['passed']}/{c['total']} ({c['passed']/c['total']*100:.0f}%)")
            print()

        # Per-task detail
        print("Tasks:")
        for r in run:
            mark = "PASS" if r["passed"] else "FAIL"
            print(f"  [{mark}] {r['task_id']}: {r['task_description']}")
            for g in r.get("grades", []):
                print(f"    {'+' if g['score'] == 1.0 else '-'} {g['type']}: {g['reason'][:120]}")
            if r.get("error"): print(f"    Error: {r['error'][:200]}")

        # Aggregate metrics
        ok = [r for r in run if not r.get("error")]
        if ok:
            print(f"\nMetrics (avg): {sum(r['metrics']['time'] for r in ok)/len(ok):.2f}s, "
                  f"{sum(r['metrics']['tool_calls'] for r in ok)/len(ok):.1f} tool calls, "
                  f"{sum(r['metrics']['turns'] for r in ok)/len(ok):.1f} turns")
            print(f"Tokens: {sum(r['metrics']['input_tokens'] for r in ok):,} in, "
                  f"{sum(r['metrics']['output_tokens'] for r in ok):,} out")
        print()


def inspect_task(results, task_id, run_index=0):
    """Print detailed results for a specific task including transcript."""
    run = results["runs"][run_index]
    r = next((r for r in run if r["task_id"] == task_id), None)
    if r is None:
        print(f"Task '{task_id}' not found"); return

    print(f"[{'PASS' if r['passed'] else 'FAIL'}] {r['task_id']}: {r['task_description']}")
    print(f"Query: {r['query']}")
    print(f"Response: {r.get('final_text', 'N/A')}\n")
    if r.get("error"): print(f"ERROR:\n{r['error']}"); return

    print("Grades:")
    for g in r["grades"]:
        print(f"  {'+' if g['score'] == 1.0 else '-'} {g['type']}: {g['reason']}")
    print(f"\nMetrics: {r['metrics']}")

    print("\nTranscript:")
    for item in r.get("transcript", []):
        if isinstance(item, dict):
            t = item.get("type", "?")
            if t == "text": print(f"  [text] {item.get('text', '')[:300]}")
            elif t == "tool_use": print(f"  [tool_use] {item.get('name', '?')}({item.get('input', {})})")
            elif t == "tool_result": print(f"  [tool_result] {str(item.get('content', ''))[:200]}")
            else: print(f"  [{t}] {str(item)[:200]}")
        else: print(f"  {str(item)[:200]}")


print("Eval framework ready.")

Eval framework ready.


### Design decisions worth noting

If you read through the grader and runner code above, a few choices stand out:

- **Registry dict, not if/elif.** Graders are looked up by type name in `GRADER_REGISTRY`, a plain dict. Adding a new grader type is one function + one dict entry, no need to touch a dispatch chain. This is the pattern most eval frameworks use.

- **The runner doesn't print.** `run_eval()` returns a data structure; `print_summary()` is a separate function. This keeps the runner testable and reusable. You could swap in a different display, save to JSON, or feed results into a dashboard without changing the runner.

- **Agent as parameter.** The runner takes `agent_fn` as an argument instead of importing the agent directly. This makes it easy to test with a mock, swap in a different agent, or wrap the agent with instrumentation.

- **Runner owns transcript parsing.** The agent returns raw messages; the runner extracts `final_text`, `tool_calls`, etc. This avoids duplicate extraction logic and keeps the agent clean. It doesn't need to know how it will be evaluated.

- **Failed runs ≠ failed tasks.** If the agent throws an exception (API error, timeout), that's a *failed run*, captured in the `error` field. Grading is skipped. If the agent returns the wrong answer, that's a *failed task* : graders run normally and report what went wrong. The distinction matters. Errors mean the infrastructure broke; failures mean the agent gave the wrong answer.

- **Concurrency.** Tasks run in parallel using `ThreadPoolExecutor`, the standard pattern for I/O-bound API calls. Unbounded parallelism would hit rate limits. If your agent used the async `AsyncAnthropic` client instead, you'd replace this with `asyncio.gather()`.

---

## Part 3: Define Your Eval Tasks

### Task schema

Each task is a Python dict with these fields:

```python
{
    "id": "unique_task_id",             # Short, descriptive identifier
    "description": "What this tests",    # Human-readable description
    "query": "The user's question",      # What gets sent to the agent
    "category": "product_lookup",        # For grouping results
    "graders": [                          # List of grader declarations
        {
            "type": "response_contains",  # Which grader to use
            "checks": ["49.99"],          # What to check (one or more)
        },
    ],
}
```

### Worked example

Here's a task that checks whether the agent can look up the price of jeans:

```python
{
    "id": "price_jeans",
    "description": "Direct price lookup for jeans",
    "query": "How much do jeans cost?",
    "category": "product_lookup",
    "graders": [
        {"type": "response_contains", "checks": ["49.99"]},
        {"type": "tool_use", "checks": [{"tool_name": "get_product", "arguments": {"product": "jeans"}}]},
    ],
}
```

This task uses two graders:
1. `response_contains` checks that the final answer mentions "49.99"
2. `tool_use` checks that the agent actually called `get_product` with `product="jeans"` (didn't hallucinate the price)

### A note on `tool_use` grader brittleness

Tool use checks should verify that the agent *grounded its answer in tools* (didn't hallucinate a price or do mental math), not enforce a rigid call sequence. If the agent reaches the correct answer via a valid but unexpected path, that's fine.

Checking `{"tool_name": "get_product"}` (no arguments) verifies the agent used the tool at all. Checking `{"tool_name": "get_product", "arguments": {"product": "jeans"}}` verifies the exact argument. Only do this when the argument value is the thing you're testing (e.g., synonym resolution).

### Your turn

The jeans task is given as a reference. Now build tasks for these five example queries:

1. `"Price of a t-shirt?"` - Will the agent handle the hyphen correctly? What argument should it pass to `get_product`?
2. `"How much for shoes?"` - "shoes" isn't in the catalog. What *should* happen here, and how do you grade it?
3. `"3 shirts and 2 belts, what's my total?"` - Multiple product lookups plus a calculation. What's the expected total?
4. `"What's 20% off a jacket?"` - Needs both a product lookup and a calculation. What number should you check for?
5. `"What do you sell?"` - Open-ended. Can you grade this with the graders you have? (Hint: come back to this one after Part 6.)

For each query, think about:
- Which category does it belong to?
- Which graders make sense? What checks?
- What's the expected correct answer?
- Is a `tool_use` check needed, or is checking the response enough?

### ✏️ YOUR TURN — write your eval tasks in THIS cell

Each task is a dict that needs an **id**, the **query** (the prompt sent to the agent), and **graders** with **checks** that decide pass/fail — keep the worked example, copy the commented template for each new task, and add yours below the `# ✏️ ADD YOUR TASKS BELOW` marker.

In [6]:
# ✏️ YOUR TURN — write your eval tasks in THIS list.
# Each task needs an id, the query (prompt) to send to the agent, and graders with checks.
#
# Completed. These are the 6 dev tasks the deterministic graders can check. The 5 that need an
# LLM judge live in Part 6. Same ids and checks as ab/tasks_dev.py, so the numbers compare with the
# A/B suite. Spec decisions behind them:
#   - close-but-not-carried ("shoes") -> suggest the closest item (sneakers) with its real price
#   - not carried at all ("piano", brands) -> say so; never invent a product or price
#   - one self-contained message per task (the agent is single-turn, no memory)
tasks = [
    # ── Reference task (worked example) ─────────────────────────────────────
    {
        "id": "price_jeans",
        "description": "Direct price lookup for jeans",
        "query": "How much do jeans cost?",
        "category": "product_lookup",
        "graders": [
            {"type": "response_contains", "checks": ["49.99"]},
            {"type": "tool_use", "checks": [{"tool_name": "get_product", "arguments": {"product": "jeans"}}]},
        ],
    },

    # 1. "Price of a t-shirt?" -> 24.99. The hyphenated key is what's being tested, so we check
    #    the exact argument.
    {
        "id": "price_tshirt",
        "description": "Hyphenated product name",
        "query": "Price of a t-shirt?",
        "category": "product_lookup",
        "graders": [
            {"type": "response_contains", "checks": ["24.99"]},
            {"type": "tool_use", "checks": [{"tool_name": "get_product", "arguments": {"product": "t-shirt"}}]},
        ],
    },

    # 2. "How much for shoes?" -> not in the catalog; spec says suggest sneakers at 74.99.
    #    The shoes->sneakers mapping is the behavior, so the argument check is the point.
    {
        "id": "synonym_shoes",
        "description": "Not in catalog but close: should suggest sneakers",
        "query": "How much for shoes?",
        "category": "synonym",
        "graders": [
            {"type": "response_contains", "checks": ["sneakers"]},
            {"type": "response_numeric", "checks": [{"value": 74.99, "tolerance": 0.01}]},
            {"type": "tool_use", "checks": [{"tool_name": "get_product", "arguments": {"product": "sneakers"}}]},
        ],
    },

    # 3. "3 shirts and 2 belts" -> 3x29.99 + 2x24.99 = 139.95. Both lookups + calculate are
    #    required so a lucky or mental-math answer can't pass.
    {
        "id": "multi_shirts_belts",
        "description": "Two lookups + math: 3x29.99 + 2x24.99 = 139.95",
        "query": "3 shirts and 2 belts, what's my total?",
        "category": "multi_step_math",
        "graders": [
            {"type": "response_numeric", "checks": [{"value": 139.95, "tolerance": 0.01}]},
            {"type": "tool_use", "checks": [
                {"tool_name": "get_product", "arguments": {"product": "shirt"}},
                {"tool_name": "get_product", "arguments": {"product": "belt"}},
                {"tool_name": "calculate"},
            ]},
        ],
    },

    # 4. "20% off a jacket" -> 89.99 x 0.8 = 71.992 -> 71.99.
    {
        "id": "discount_jacket",
        "description": "Percentage math: 89.99 x 0.8 = 71.99",
        "query": "What's 20% off a jacket?",
        "category": "multi_step_math",
        "graders": [
            {"type": "response_numeric", "checks": [{"value": 71.99, "tolerance": 0.01}]},
            {"type": "tool_use", "checks": [
                {"tool_name": "get_product", "arguments": {"product": "jacket"}},
                {"tool_name": "calculate"},
            ]},
        ],
    },

    # 5. "What do you sell?" -> open-ended, can't be string-matched: see Part 6.

    # ✏️ ADD YOUR TASKS BELOW
    # From the chat session: the shipped agent answered "jean? how much?" by asking about brands.
    # Known grader caveat (kept on purpose): with forgiving matching, get_product("jean") also
    # returns 49.99, and the strict argument check would fail that valid path.
    {
        "id": "singular_jean",
        "description": "Casual singular wording ('jean')",
        "query": "jean? how much?",
        "category": "product_lookup",
        "graders": [
            {"type": "response_numeric", "checks": [{"value": 49.99, "tolerance": 0.01}]},
            {"type": "tool_use", "checks": [{"tool_name": "get_product", "arguments": {"product": "jeans"}}]},
        ],
    },
]

---

## Part 4: Run the Eval

Now let's run the eval and see how the agent performs. Results are also saved as JSON files so you can compare across runs.

> **Completed notebook:** the agent in Part 1 is already the improved version (see Part 5), so these runs show the *after* numbers. The before/after comparison is in **Findings** below.


In [7]:
results = run_eval(run_agent, tasks)
print_summary(results)
save_results(results)

  [1/6] price_jeans: PASS


  [2/6] synonym_shoes: PASS


  [3/6] singular_jean: PASS


  [4/6] price_tshirt: PASS


  [5/6] multi_shirts_belts: PASS


  [6/6] discount_jacket: PASS


EVAL RESULTS: 6 tasks, 1 run(s)

Overall: 6/6 passed (100%)

By category:
  multi_step_math: 2/2 (100%)
  product_lookup: 3/3 (100%)
  synonym: 1/1 (100%)

Tasks:
  [PASS] price_jeans: Direct price lookup for jeans
    + response_contains: Found '49.99' in response
    + tool_use: Tool 'get_product' called with matching args: {'product': 'jeans'}
  [PASS] price_tshirt: Hyphenated product name
    + response_contains: Found '24.99' in response
    + tool_use: Tool 'get_product' called with matching args: {'product': 't-shirt'}
  [PASS] synonym_shoes: Not in catalog but close: should suggest sneakers
    + response_contains: Found 'sneakers' in response
    + response_numeric: Found 74.99 (expected 74.99 +/- 0.01)
    + tool_use: Tool 'get_product' called with matching args: {'product': 'sneakers'}
  [PASS] multi_shirts_belts: Two lookups + math: 3x29.99 + 2x24.99 = 139.95
    + response_numeric: Found 139.95 (expected 139.95 +/- 0.01)
    + tool_use: Tool 'get_product' called with match

'eval_results/eval_default_20261113_102532.json'

### Inspecting results

The summary tells you *what* passed and failed. To understand *why*, inspect individual tasks.

For any failed task, look at:
- What did the agent actually respond with?
- What tool calls did it make? With what arguments?
- Where did it go wrong: tool selection, argument formatting, or the final answer?

For passing tasks, ask: is it passing for the right reasons? Could it be passing by luck?

For failing tasks, ask: does it feel fair?

In [8]:
# Replace with a task ID you want to inspect
inspect_task(results, "price_jeans")

[PASS] price_jeans: Direct price lookup for jeans
Query: How much do jeans cost?
Response: Jeans cost $49.99.

Grades:
  + response_contains: Found '49.99' in response
  + tool_use: Tool 'get_product' called with matching args: {'product': 'jeans'}

Metrics: {'time': 1.67, 'tool_calls': 1, 'turns': 2, 'input_tokens': 2073, 'output_tokens': 62}

Transcript:
  How much do jeans cost?
  [tool_use] get_product({'product': 'jeans'})
  [tool_result] 49.99
  [text] Jeans cost $49.99.


### Establishing a baseline

LLMs are non-deterministic — the newest Claude models don't expose sampling parameters like temperature at all, and pinned settings never guaranteed identical outputs anyway. Run the eval multiple times to get a reliable baseline before making changes.

In [9]:
baseline = run_eval(run_agent, tasks, num_runs=5)
print_summary(baseline)

  [1/6] price_jeans: PASS


  [2/6] price_tshirt: PASS


  [3/6] synonym_shoes: PASS


  [4/6] singular_jean: PASS


  [5/6] discount_jacket: PASS


  [6/6] multi_shirts_belts: PASS


... (21 outputs left out)

  [4/6] singular_jean: PASS


  [5/6] multi_shirts_belts: PASS


  [6/6] discount_jacket: PASS


EVAL RESULTS: 6 tasks, 5 run(s)

--- Run 1 ---
Overall: 6/6 passed (100%)

By category:
  multi_step_math: 2/2 (100%)
  product_lookup: 3/3 (100%)
  synonym: 1/1 (100%)

Tasks:
  [PASS] price_jeans: Direct price lookup for jeans
    + response_contains: Found '49.99' in response
    + tool_use: Tool 'get_product' called with matching args: {'product': 'jeans'}
  [PASS] price_tshirt: Hyphenated product name
    + response_contains: Found '24.99' in response
    + tool_use: Tool 'get_product' called with matching args: {'product': 't-shirt'}
  [PASS] synonym_shoes: Not in catalog but close: should suggest sneakers
    + response_contains: Found 'sneakers' in response
    + response_numeric: Found 74.99 (expected 74.99 +/- 0.01)
    + tool_use: Tool 'get_product' called with matching args: {'product': 'sneakers'}
  [PASS] multi_shirts_belts: Two lookups + math: 3x29.99 + 2x24.99 = 139.95
    + response_numeric: Found 139.95 (expected 139.95 +/- 0.01)
    + tool_use: Tool 'get_product' cal

---

## Part 5: Improve the Agent

Use what you learned from the eval results to improve the agent.

Things you can change:
- **Tool descriptions.** The current specs are intentionally vague. What information would help Claude use the tools correctly?
- **System prompt.** `"You are a helpful assistant."` gives Claude zero context about being a shopping assistant.
- **Error handling.** What happens when a product isn't found? When an invalid operator is used?
- **Tool implementation.** Could the tool be more forgiving about input format?

After making changes, **scroll back up and re-run the agent cell**, then re-run the eval to see if your changes helped.

> **Already applied in this notebook.** The agent cell in Part 1 now holds the best measured variant from the A/B suite (`ab/`). To redo the exercise from scratch, copy the agent cell from `ab/shipped/Building_an_Eval.shipped.ipynb`. The evidence for each change is in `ab/reflection.md`.


---

## Part 6: Add an LLM-as-Judge Grader

Some queries can't be graded with a deterministic grader. Consider:
- *"What do you sell?"* There are many valid ways to describe the catalog.
- *"Which is a better deal, 2 shirts or 1 jacket?"* The answer requires reasoning, not just a number.
- *"I have $100, what should I buy?"* Any sensible recommendation is a valid answer.

For these, we need an **LLM-as-judge**: a grader that uses Claude itself to evaluate the response.

### The contract

The grader interface is the same as the deterministic graders:

```python
def grade_llm_judge(result, check, context=None):
    # check: a string describing what to evaluate, e.g.
    #   "Response lists available product categories"
    #   "Response correctly identifies the better deal with reasoning"
    #
    # context: includes the original query (context["query"]) and task metadata
    #
    # Returns: {"score": 0.0 or 1.0, "reason": "..."}
```

Each check is a natural-language criterion. The grader sends the agent's response + the criterion to Claude and asks for a binary pass/fail judgment.

### Implementation hints

- **Prompt structure.** Give the judge the original query, the agent's response (`result["final_text"]`), and the criterion (`check`). Ask it to evaluate whether the criterion is met.
- **Force a structured answer.** Ask the judge to respond with `PASS` or `FAIL` on the first line, followed by a reason. This makes parsing straightforward.
- **Keep it atomic.** Each check is a single criterion. Don't ask one LLM call to evaluate multiple criteria at once.

### Implement it

Fill in the grader below, register it, then write tasks that use it.

In [10]:
# Implement the LLM-as-judge grader
#
# We use structured outputs (output_config.format) so the verdict comes back as
# validated JSON with an enum — no string-parsing a free-text "PASS"/"FAIL", and
# no sampling params (temperature is removed on the newest models; determinism
# comes from the schema, not the dial).

JUDGE_SCHEMA = {
    "type": "json_schema",
    "schema": {
        "type": "object",
        "properties": {
            # reason before verdict: the judge explains first, then commits
            "reason": {"type": "string", "description": "One sentence, citing the response."},
            "verdict": {"type": "string", "enum": ["PASS", "FAIL"]},
        },
        "required": ["reason", "verdict"],
        "additionalProperties": False,
    },
}

JUDGE_SYSTEM = (
    "You grade one response from a shopping assistant against ONE criterion. "
    "Judge only that criterion: ignore tone, length, and anything the criterion doesn't mention. "
    "Use any ground truth given in the criterion; don't rely on outside knowledge of prices. "
    "If the response is empty or doesn't address the criterion, the verdict is FAIL. "
    "Write the reason first, then the verdict."
)


def grade_llm_judge(result, check, context=None):
    # Step 1: the judge sees the query, the agent's final answer, and ONE criterion
    query = (context or {}).get("query", "")
    prompt = (
        f"<customer_query>\n{query}\n</customer_query>\n\n"
        f"<assistant_response>\n{result['final_text']}\n</assistant_response>\n\n"
        f"<criterion>\n{check}\n</criterion>"
    )
    # Step 2: a structured verdict from Haiku 5.5 (the judge stays fixed so the ruler never moves)
    response = client.messages.create(
        model=FAST_MODEL,
        max_tokens=4000,  # Haiku 5.5 thinks first; leave room before the JSON
        system=JUDGE_SYSTEM,
        messages=[{"role": "user", "content": prompt}],
        output_config={"format": JUDGE_SCHEMA},
    )
    # Step 3: read the verdict — skip the thinking block, which comes first on Haiku 5.5
    text = next(b.text for b in response.content if b.type == "text")
    data = json.loads(text)
    return {"score": 1.0 if data["verdict"] == "PASS" else 0.0, "reason": f"judge: {data['reason']}"}


# Register it so the runner can use it
GRADER_REGISTRY["llm_judge"] = grade_llm_judge

# Grade the grader: two answers whose verdict we already know. If these disagree, fix the judge
# before trusting any score it gives.
_crit = "The response does not state or estimate any price for a piano."
for _text, _want in [("We don't sell pianos, sorry!", "PASS"),
                     ("We don't sell pianos, but they usually run around $3,000.", "FAIL")]:
    _g = grade_llm_judge({"final_text": _text}, _crit, {"query": "How much is a piano?"})
    print(f"judge sanity: want {_want}, got {'PASS' if _g['score'] else 'FAIL'} — {_g['reason']}")

judge sanity: want PASS, got PASS — judge: The response states no price or estimate for a piano, only saying the store doesn't sell pianos, so the criterion is satisfied.


judge sanity: want FAIL, got FAIL — judge: The response states a price estimate of around $3,000 for a piano, which violates the criterion that it must not state or estimate any piano price.


In [11]:
# Add tasks that use the LLM-as-judge grader
# Ground truth goes INTO the criterion text: the judge only sees the query and the answer.
# One criterion per check, so each verdict is about one thing.

CATALOG_PRICES = ("jeans 49.99, shirt 29.99, dress 59.99, jacket 89.99, sneakers 74.99, hat 19.99, "
                  "socks 9.99, hoodie 44.99, shorts 34.99, t-shirt 24.99, sweater 54.99, belt 24.99")

llm_judge_tasks = [
    # 5. from Part 3: "What do you sell?"
    {
        "id": "catalog_overview",
        "description": "Open-ended: describe what the shop sells",
        "query": "What do you sell?",
        "category": "catalog",
        "graders": [
            {"type": "llm_judge", "checks": [
                f"The response names at least four products from this catalog: {CATALOG_NAMES}.",
                f"The response does not offer any product or category outside this catalog "
                f"({CATALOG_NAMES}), such as electronics, home goods, or brands.",
            ]},
        ],
    },
    # From the chat session: budget math, brand we don't carry, off-catalog item, recommendation.
    {
        "id": "budget_4_jeans",
        "description": "Budget check: 4 x 49.99 = 199.96 > 100",
        "query": "I have $100 and need 4 jeans. Can I afford it?",
        "category": "multi_step_math",
        "graders": [
            {"type": "response_numeric", "checks": [{"value": 199.96, "tolerance": 0.01}]},
            {"type": "tool_use", "checks": [{"tool_name": "calculate"}]},
            {"type": "llm_judge", "checks": [
                "The response clearly tells the customer that 4 pairs of jeans cost more than their $100 budget.",
            ]},
        ],
    },
    {
        "id": "brand_not_carried",
        "description": "Brand we don't carry: no invented Levi's, offer our jeans",
        "query": "Do you have Levi's jeans?",
        "category": "negative",
        "graders": [
            {"type": "response_numeric", "checks": [{"value": 49.99, "tolerance": 0.01}]},
            {"type": "llm_judge", "checks": [
                "The response does not claim the store carries Levi's or any other named brand, "
                "and does not invent Levi's product lines or styles.",
            ]},
        ],
    },
    {
        "id": "off_catalog_piano",
        "description": "Totally off-catalog: say not carried, no price",
        "query": "How much is a piano?",
        "category": "negative",
        "graders": [
            {"type": "llm_judge", "checks": [
                "The response tells the customer the store does not sell pianos.",
                "The response does not state or estimate any price for a piano.",
            ]},
        ],
    },
    {
        "id": "best_value_100",
        "description": "Open-ended recommendation under a budget",
        "query": "I have $100 and need at least 1 jeans and 1 shirt. What's the best value I can get?",
        "category": "recommendation",
        "graders": [
            {"type": "llm_judge", "checks": [
                "The recommended purchase includes at least one pair of jeans and at least one shirt "
                "(the catalog item 'shirt' or 't-shirt' both count as a shirt).",
                f"The total cost of the recommended purchase is $100.00 or less, computed with these "
                f"catalog prices: {CATALOG_PRICES}.",
                f"Every price the response quotes matches these catalog prices: {CATALOG_PRICES}.",
            ]},
        ],
    },
]

# Run eval with both task sets: the full 11-task dev set
all_tasks = tasks + llm_judge_tasks
results = run_eval(run_agent, all_tasks, num_runs=3)
print_summary(results)
save_results(results)

  [1/11] price_jeans: PASS


  [2/11] price_tshirt: PASS


  [3/11] synonym_shoes: PASS


  [4/11] singular_jean: PASS


  [5/11] multi_shirts_belts: PASS


  [6/11] discount_jacket: PASS


... (25 outputs left out)

  [10/11] off_catalog_piano: PASS


  [11/11] best_value_100: PASS


EVAL RESULTS: 11 tasks, 3 run(s)

--- Run 1 ---
Overall: 11/11 passed (100%)

By category:
  catalog: 1/1 (100%)
  multi_step_math: 3/3 (100%)
  negative: 2/2 (100%)
  product_lookup: 3/3 (100%)
  recommendation: 1/1 (100%)
  synonym: 1/1 (100%)

Tasks:
  [PASS] price_jeans: Direct price lookup for jeans
    + response_contains: Found '49.99' in response
    + tool_use: Tool 'get_product' called with matching args: {'product': 'jeans'}
  [PASS] price_tshirt: Hyphenated product name
    + response_contains: Found '24.99' in response
    + tool_use: Tool 'get_product' called with matching args: {'product': 't-shirt'}
  [PASS] synonym_shoes: Not in catalog but close: should suggest sneakers
    + response_contains: Found 'sneakers' in response
    + response_numeric: Found 74.99 (expected 74.99 +/- 0.01)
    + tool_use: Tool 'get_product' called with matching args: {'product': 'sneakers'}
  [PASS] multi_shirts_belts: Two lookups + math: 3x29.99 + 2x24.99 = 139.95
    + response_numeric: F

'eval_results/eval_default_20261113_102652.json'

---

## Findings

The full evidence trail is in [`ab/reflection.md`](ab/reflection.md): the A/B suite, 20+ variants, a blind holdout set, grader tests, and the model sweep. Here's the short version, then a live before/after on the same 11 tasks.

### Scoreboard (Haiku 5.5, from `ab/`)

| | dev (11 tasks) | blind holdout (11 tasks) | $/query |
|---|---|---|---|
| Shipped agent | 52% | 42% | $0.0003 |
| **Improved agent (this notebook)** | **100%** (5 trials) | **100%** (3 trials) | **$0.0004** |

### What we learned

1. **It's not the model.** On the shipped agent, every model tested tops out at 52–73%, including Sonnet 5.5. The same tasks fail everywhere: shoes→sneakers, "what do you sell?" and "how much is a piano?". No model can know a catalog it's never shown. Asked about a piano, Sonnet 5.5 reported *"the lookup returned an error both times"*, then guessed prices anyway.
2. **Root cause: the agent doesn't know where it works.** `"You are a helpful assistant."` and an invisible catalog explain the failures we saw in chat: asking the customer for prices, asking about brands and currency, inventing Levi's lines.
3. **Two prompt changes do the whole job.** The shop context plus the product names score 100% on dev *and* holdout. The tool fixes (helpful errors, forgiving matching, a calculator that errors on bad input) didn't move the score. They're kept as cheap hardening against failures this eval doesn't test.
4. **Catalog in the prompt beat a `list_products` tool:** same score, ~15% cheaper. After listing products, the agent also tended to skip `calculate`.
5. **Haiku 5.5 is the right model here.** With the fix it ties Sonnet 5.5 at 100%, at about 1/20 the cost.
6. **Grade the graders first.** In round 1, 9 of 57 failures were scoring errors, not agent errors:
   - a strict argument match rejected `get_product("jean")` → 49.99,
   - the runner graded only the last text block (fixed in Part 2's runner above),
   - checks were stricter than the spec.

   Without the grader tests, we'd have concluded that forgiving matching makes the agent worse. It doesn't.
7. **Re-run evals when you change models.** Better tool descriptions were necessary on Haiku 4.5 and redundant on 5.5.

### Before vs after, live

The cell below runs the **shipped** agent (frozen in `ab/shipped/`) and the **improved** agent on the same 11 tasks, with the same model (Haiku 5.5), the same graders and the same judge. Only the agent changes.

In [12]:
# Before vs after: same tasks, same model (Haiku 5.5), same graders + judge — only the agent differs
sys.path.insert(0, str(pathlib.Path.cwd() / "ab"))
from nbload import fresh_namespace

shipped = fresh_namespace()  # the agent exactly as shipped, frozen in ab/shipped/
before = run_eval(shipped["run_agent"], all_tasks, model=FAST_MODEL, num_runs=3)
after = results  # Part 6 run of the improved agent

def per_task(res):
    out = {}
    for run in res["runs"]:
        for r in run:
            out.setdefault(r["task_id"], []).append(r["passed"])
    return out

b, a = per_task(before), per_task(after)
print(f"\n{'task':<22}{'shipped':>10}{'improved':>10}")
print("-" * 42)
for t in all_tasks:
    tid = t["id"]
    sb, sa = f"{sum(b[tid])}/{len(b[tid])}", f"{sum(a[tid])}/{len(a[tid])}"
    print(f"{tid:<22}{sb:>10}{sa:>10}")
tb = sum(map(sum, b.values())); ta = sum(map(sum, a.values())); n = sum(map(len, a.values()))
print("-" * 42)
print(f"{'TOTAL':<22}{f'{tb}/{n}':>10}{f'{ta}/{n}':>10}")
print(f"{'rate':<22}{tb/n:>10.0%}{ta/n:>10.0%}")

  [1/11] price_jeans: PASS


  [2/11] discount_jacket: FAIL


  [3/11] price_tshirt: PASS


  [4/11] synonym_shoes: FAIL


  [5/11] singular_jean: PASS


  [6/11] multi_shirts_belts: PASS


... (24 outputs left out)

  [9/11] brand_not_carried: FAIL


  [10/11] off_catalog_piano: FAIL


  [11/11] best_value_100: PASS



task                     shipped  improved
------------------------------------------
price_jeans                  3/3       3/3
price_tshirt                 3/3       3/3
synonym_shoes                0/3       3/3
multi_shirts_belts           2/3       3/3
discount_jacket              0/3       3/3
singular_jean                3/3       3/3
catalog_overview             0/3       3/3
budget_4_jeans               2/3       3/3
brand_not_carried            1/3       3/3
off_catalog_piano            0/3       3/3
best_value_100               3/3       3/3
------------------------------------------
TOTAL                      17/33     33/33
rate                         52%      100%


---

## Extensions

Finished early? Here are ideas ordered roughly by complexity.

### Quick wins
- **Negative test cases.** Queries the agent *should* handle gracefully: "How much is a piano?" (not in catalog), "What's the meaning of life?" (off-topic).
- **Task validation.** Write a function that checks every task dict has the required fields before running the eval.
- **Generate tasks with Claude.** Ask Claude Code to generate additional eval tasks based on the catalog and the patterns you've established.

### Model comparison
- **Haiku vs Sonnet.** This notebook runs Haiku 5.5 only. The cross-model comparison lives in the A/B suite: `uv run python ab/ab_suite.py v0_baseline v_all_catprompt --model claude-sonnet-5-5`. Result: a stronger model can't make up for a missing catalog, and once fixed, both models score 100%.
- **Cost vs accuracy.** Aggregate token usage and estimated cost per run. Is Sonnet worth the extra cost if Haiku passes 90% of tasks? (In `ab/`: both score 100% on the fixed agent; Haiku 5.5 is ~20x cheaper.)

### Better graders
- **Efficiency graders.** Grade not just correctness but efficiency: did the agent use the minimum number of tool calls?
- **Grader auto-discovery.** Replace the manual `GRADER_REGISTRY` dict with a `@grader` decorator that registers functions automatically.

### Analysis and visualization
- **Results visualization.** Build a chart (matplotlib, plotly) showing pass rates by category or a comparison across models.
- **Baseline comparison.** Diff two saved JSON results side-by-side, flag regressions.
- **Pass@k and pass^k.** Run each task k times. Pass@k = "passed at least once"; pass^k = "passed every time". Useful for distinguishing flaky from reliably broken.
- **Statistical robustness.** Compute variance, confidence intervals, and statistical significance across runs.

### Harness improvements
- **Max turns guard.** Prevent the agent from looping forever by adding a turn limit.
- **Per-task timeouts.** Kill tasks that take too long.
- **Retry with backoff.** Handle transient API errors gracefully.
- **Caching.** Avoid re-running unchanged tasks by hashing the query + agent config.
- **Progress tracking.** Show a live progress bar as the eval runs.
- **Parameter sweep.** Cartesian product over prompts, effort levels, models, etc.

---

## Beyond This Session

### What we simplified

Some design choices in this eval are intentionally simple. Here's what production systems do differently:

- **Agent instrumentation.** We added an `eval_mode` flag so the agent returns the full transcript. In practice, you don't want to modify the agent at all. Production frameworks wrap the agent and intercept API calls to build the transcript without touching agent code.

- **Observability.** We log the full transcript as a flat list. Production systems use OpenTelemetry span trees to capture structured traces with timing, nesting, and metadata, so you can see not just *what* happened but *how long* each step took and how calls were nested.

- **Environment isolation.** Our agent runs in the same process as the eval. For evals that run code (like SWE-bench), a 3-layer Docker pattern is the standard: (1) base image with OS + runtime, (2) environment image with dependencies installed (content-addressable via hash, shared across tasks), (3) instance image with the specific repo checkout per task.

- **Task format.** We use inline Python dicts. Production harnesses typically use JSONL files (one task per line), YAML with references to external files, or database-backed task stores.

### Existing frameworks

You don't have to build an eval harness from scratch. Several frameworks provide this out of the box:

- **Promptfoo.** Open-source eval framework with a nice CLI and web UI.
- **Braintrust.** Hosted eval platform with logging, tracing, and comparison tools.
- **LangSmith.** Tracing and eval platform from the LangChain ecosystem.
- **Harbor.** Open-source eval framework focused on LLM safety.

Internally, Anthropic uses **eval-toolbox** to add evals to our **eval-registry**. A natural follow-up to this session would be migrating your eval to one of these frameworks.